# Fase 2 — Forecast baseline y punto de reorden

Este notebook cierra el MVP del proyecto con:
1. Clasificación ABC por contribución de ventas
2. Features rolling de demanda por SKU
3. Backtest temporal baseline (MAE/MAPE)
4. Tabla final de reorden por producto

In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    clean_transactions,
    sales_by_product_last_quarter,
    compute_abc_classification,
    build_daily_sku_demand,
    build_rolling_features,
)
from inventario_ecommerce.modeling.train import temporal_backtest_baseline
from inventario_ecommerce.modeling.predict import forecast_30d_baseline, build_reorder_policy

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
raw = load_transactions()
clean = clean_transactions(raw)
daily = build_daily_sku_demand(clean)
rolling = build_rolling_features(daily)

latest_features = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print(f"Raw rows: {len(raw):,}")
print(f"Clean rows: {len(clean):,}")
print(f"Daily rows: {len(daily):,}")
print(f"SKUs: {latest_features[config.COL_STOCK_CODE].nunique():,}")

In [ ]:
last_q = sales_by_product_last_quarter(clean)
abc = compute_abc_classification(last_q)
abc["ABCClass"].value_counts()

In [ ]:
sku_metrics, global_metrics = temporal_backtest_baseline(daily, horizon_days=30, lookback_days=30)
global_metrics

In [ ]:
forecast = forecast_30d_baseline(daily, lookback_days=30, horizon_days=30)
policy = build_reorder_policy(latest_features, forecast, abc)
policy.head(20)

In [ ]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest_features, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")
save_processed(policy, "inventory_reorder_recommendations.csv")

print("Artefactos guardados en:", config.PROCESSED_DATA_DIR)

## Entregable final

Tabla principal: `data/processed/inventory_reorder_recommendations.csv`

Campos clave: `ABCClass`, `forecast_30d`, `safety_stock`, `reorder_point`, `target_stock`.